# Analysis of partially correct ICL demonstrations

This notebook reproduces the results of section 4.2 of the paper.
Before running this notebook you need:
1. Results of in-context NED for all dataset, perturbation types, and perturbation factors.
2. Results of zero-shot and 10-shot NED for all datasets.

For more informatin on on how to run these experiments, please refer to the README file.

In [ ]:
import ast
import json
import os

import matplotlib.pyplot as plt
import pandas as pd
from hydra import compose, initialize
from matplotlib.lines import Line2D
from omegaconf import OmegaConf
from utils.evaluation import evaluate_entities

In [ ]:
seed = 12345
results_folder = "outputs"
dataset_list = ['chemprotgene', 'chemprotchem', 'bc5chem', 'bc5disease', 'bc2gm']
perturbation_types = ['addition-substitution', 'deletion-substitution', 'deletion', 'substitution']
perturbation_factors = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
demo_retrieval='knn'

Aggregate the results for all perturbation experiments, Deletion, Substitution, Addition and Substitution, Deletion and Substitution, and perturbation factors, 0.1-0.9, for all datasets, 'chemprotgene', 'chemprotchem', 'bc5chem', 'bc5disease', 'bc2gm'.

In [ ]:
setups = []
for dataset in dataset_list:
    for perturbation in perturbation_types:
        for factor in perturbation_factors:
            setups.append({
                'dataset': dataset,
                'perturbation': perturbation,
                'factor': factor,
                'results_path': ''
            })

sorted_experiments_desc = sorted(os.listdir(results_folder), reverse=True)
for day in sorted_experiments_desc:
    for time in sorted(os.listdir(os.path.join(results_folder, day)), reverse=True):
        with initialize(config_path=os.path.join(results_folder, day, time, '.hydra')):
            cfg = compose(config_name="config")
        
        cfg_dict = OmegaConf.to_container(cfg, resolve=True)
        dataset = cfg_dict['data']['dataset']
        perturbation = cfg_dict.get('perturbation_type', {}).get('name', None)
        factor = cfg_dict.get('perturbation_factor', {}).get('value', None)

        if dataset in dataset_list and perturbation in perturbation_types and factor in perturbation_factors and cfg_dict['demonstration_retrieval'].get('method', None) == demo_retrieval:
            for setup in setups:
                if setup['dataset'] == dataset and setup['perturbation'] == perturbation and setup['factor'] == factor and setup['results_path'] == '':
                    setup['results_path'] = os.path.join(results_folder, day, time, 'results.csv')
                    break

for setup in setups:
    if setup['results_path'] == '':
        print(f"Missing results for {setup['dataset']} {setup['perturbation']} {setup['factor']}")
        continue

dataset2results = {}
for setup in setups:
    dataset = setup['dataset']
    if dataset not in dataset2results:
        dataset2results[dataset] = {"type": [], "factor": [], "id": [], "demo_prc": [], "demo_rec": [], "demo_f1":[], "demo_num_entities": [], "demo_num_entities_diff(pert-gt)": [], "pred_prc": [], "pred_rec": [], "pred_f1": []}
    
    results_df = pd.read_csv(setup['results_path'])
    
    df_demo_f1 = []
    df_pred_prc = []
    df_pred_rec = []
    df_pred_f1 = []

    results_df["processed_pred"] = results_df["processed_pred"].apply(ast.literal_eval)
    results_df["gt_entities"] = results_df["gt entities"].apply(ast.literal_eval)

    for row_id, row in enumerate(results_df.iterrows()):
        row = row[1]
        predictions = row["processed_pred"]
        gt = row["gt_entities"]
    
        prec, rec, f1, tp, fp, fn, tp_score = evaluate_entities(predictions, gt)
        dataset2results[dataset]["id"].append(row_id)
        dataset2results[dataset]["type"].append(setup['perturbation'].replace("_", " & "))
        dataset2results[dataset]["factor"].append(setup['factor'])
        dataset2results[dataset]["demo_prc"].append(row["demo_prc"])
        dataset2results[dataset]["demo_rec"].append(row["demo_rec"])
        dataset2results[dataset]["demo_num_entities"].append(row['demo_num_entities'])
        dataset2results[dataset]["demo_num_entities_diff(pert-gt)"].append(row['demo_num_entities_diff(pert-gt)'])

        demo_f1  = 0
        if row["demo_prc"] == 1 and row["demo_rec"] == 1:
            print(row)
        if row["demo_prc"] + row["demo_rec"] == 0:
            demo_f1 = 0
        else:
            demo_f1 = 2*row["demo_prc"]*row["demo_rec"]/(row["demo_prc"]+row["demo_rec"])

        dataset2results[dataset]["demo_f1"].append(demo_f1)
        dataset2results[dataset]["pred_prc"].append(prec)
        dataset2results[dataset]["pred_rec"].append(rec)
        dataset2results[dataset]["pred_f1"].append(f1)  

        df_demo_f1.append(demo_f1)
        df_pred_prc.append(prec)
        df_pred_rec.append(rec)
        df_pred_f1.append(f1)

dataset2results_df = {}
for dataset in dataset_list:
    dataset2results_df[dataset] = pd.DataFrame(dataset2results[dataset])

json.dump(dataset2results, open(f"analysis_partially_correct_demos_{demo_retrieval}_retrieval_{seed}_seed.json", "w"))

analysis_summary = []
for dataset in dataset_list:
    dataset_summary = dataset2results_df[dataset].drop(columns=["id"]).groupby(["type", "factor"]).mean()
    dataset_summary['dataset'] = [dataset]*len(dataset_summary)
    analysis_summary.append(dataset_summary)

analysis_summary = pd.concat(analysis_summary)
analysis_summary.to_csv(f"analysis_summary_partially_correct_demos_{demo_retrieval}_retrieval_{seed}_seed.csv")

Aggregate results for zero-shot and 10-shot with demonstrations drawn from full training set of the dataset, across all datasets, 'chemprotgene', 'chemprotchem', 'bc5chem', 'bc5disease', 'bc2gm'.

In [ ]:
setups = []
for dataset in dataset_list:
    for num_shots in [0,10]:
        setups.append({
            'dataset': dataset,
            'num_shots': num_shots,
            'results_path': ''
        })

sorted_experiments_desc = sorted(os.listdir(results_folder), reverse=True)
for day in sorted_experiments_desc:
    for time in os.listdir(os.path.join(results_folder, day)):
        with initialize(config_path=os.path.join(results_folder, day, time, '.hydra')):
            cfg = compose(config_name="config")
        
        cfg_dict = OmegaConf.to_container(cfg, resolve=True)
        dataset = cfg_dict['data']['dataset']
        num_shots = cfg_dict['demonstration_retrieval'].get('num_shots', None)
        demo_dataset = cfg_dict['data'].get('demo_data_filename', None)
        perturbation_type = cfg_dict.get('perturbation_type', {}).get('name', None)

        if perturbation_type in None and demo_dataset in ['train.json', 'train_sentence.json'] and dataset in dataset_list and num_shots in [0, 10]:
            for setup in setups:
                if setup['dataset'] == dataset and setup['num_shots'] == num_shots and setup['results_path'] == '':
                    setup['results_path'] = os.path.join(results_folder, day, time, 'ner_iob_evaluation.json')
                    break

baseline_results = {
    'dataset': [],
    'num_shots': [],
    'pred_prc': [],
    'pred_rec': [],
    'pred_f1': []
}
for setup in setups:
    dataset = setup['dataset']
    num_shots = setup['num_shots']
    evaluation_res = json.load(open(setup['results_path'], 'r'))
    baseline_results['dataset'].append(dataset)
    baseline_results['num_shots'].append(num_shots)
    baseline_results['pred_prc'].append(evaluation_res['micro avg']['precision'])
    baseline_results['pred_rec'].append(evaluation_res['micro avg']['recall'])
    baseline_results['pred_f1'].append(evaluation_res['micro avg']['f1-score'])

baseline_results_df = pd.DataFrame(baseline_results)
baseline_results_df.to_csv(f"baseline_results_{demo_retrieval}_retrieval.csv")

Plot the results for all datasets, perturbation types, and perturbation factors, for in-context NED, zero-shot and 10-shot NED.

In [ ]:
df = pd.read_csv(f"analysis_summary_partially_correct_demos_{demo_retrieval}_retrieval_{seed}_seed.csv")
df_avg = df.drop(columns=['dataset']).groupby(['factor', 'type']).mean().reset_index()


metric2label = {
    "f1": "Micro F1",
    "prc": "Precision",
    "rec": "Recall",
    'demo nb_entities': 'Number of entities',
    'factor': 'Perturbation factor',
    'demo_num_entities_diff(pert-gt)': 'Number of demo entities - gold entities',
}

def plot_scatter(df, title, demo_metric_list, pred_metric_list):

    gold_zero_df = pd.read_csv(f"baseline_results_{demo_retrieval}_retrieval.csv")
    gold_zero_df['type'] = gold_zero_df['type'].str.replace("_", "-")
    gold_zero_df['type'] = gold_zero_df['type'].str.replace("shot,", "shots,")
    gold_zero_df = gold_zero_df.drop(columns=['dataset'])

    fig, ax = plt.subplots(len(pred_metric_list), len(demo_metric_list), figsize=(30, 10), squeeze=False, sharey=False, sharex=True)
    for i, pred_metric in enumerate(pred_metric_list):
        for j, demo_metric in enumerate(demo_metric_list):
            
            gold_zero_aggreg = gold_zero_df.groupby(['type']).mean()[[f'pred_{pred_metric}']]
            types = df['type'].unique()
            markers = ['o', 's', 'D', '^', 'v', '<', '>', 'p', '*', '+']
            colors = ['darkcyan', 'darkorange', 'cornflowerblue', 'orangered'][:len(types)]

            for t, marker, color in zip(types, markers[:len(types)], colors):
                subset = df[df['type'] == t]
                ax[i][j].scatter(subset[f'demo_{demo_metric}'], subset[f'pred_{pred_metric}'], alpha=0.7, label=t, marker='o', s=subset['demo_num_entities']*200, color=color)

            ax[i][j].set_xlabel(f'Demonstration {metric2label[demo_metric]}', fontsize=35)
            if j == 0:
                ax[i][j].set_ylabel(f'Prediction {metric2label[pred_metric]}', fontsize=35)
            else:
                ax[i][j].set_ylabel("")  

            ax[i][j].plot([df[f'demo_{demo_metric}'].min(), df[f'demo_{demo_metric}'].max()], [df[f'demo_{demo_metric}'].min(), df[f'demo_{demo_metric}'].max()], color='black', linestyle='dotted', label='y=x', linewidth=3)
            additional_legend_handles = [Line2D([0], [0], linestyle='dotted', color='black', label='y=x')]

            line_legend_handles = []
            ax[i][j].axhline(y=gold_zero_aggreg.iloc[0][0], linestyle='--', color='black', linewidth=3)
            ax[i][j].axhline(y=gold_zero_aggreg.iloc[1][0], linestyle='-', color='black', linewidth=3)
            
            # Create legend handle for this line
            line_legend_handles.append(Line2D([0], [0], linestyle='--', color='black', label=gold_zero_aggreg.iloc[0].name))
            line_legend_handles.append(Line2D([0], [0], linestyle='-', color='black', label=gold_zero_aggreg.iloc[1].name))

            # Create custom legend handles for the types
            type_legend_handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=color, 
                                        markersize=20, label=t) for t, marker, color in zip(types, markers[:len(types)], colors)]
            all_legend_handles = type_legend_handles + line_legend_handles + additional_legend_handles

            ax[i][j].grid(color = 'gray', linestyle = '--', linewidth = 0.3)

            ax[i][j].set_ylim([-0.05, 0.75])
            ax[i][j].tick_params(axis='x', labelsize=28)
            ax[i][j].tick_params(axis='y', labelsize=28)
            ax[i][j].set_aspect('equal')

    fig.legend(handles=all_legend_handles,  loc='right', bbox_to_anchor=(1.12, 0.5), ncol=1, fontsize=30)
    
    plt.subplots_adjust(wspace=0.12, hspace=0)
    plt.title('')
    plt.savefig(title+'.pdf', bbox_inches='tight', dpi=300)
    plt.show()

In [ ]:
plot_scatter(df_avg, f'Demo micro-f1 vs. Pred micro-f1 per perturbation type using {demo_retrieval}_fulltest-sidelegend', ['f1', 'prc', 'rec'], ['f1'])